In [1]:
#Primero, hay que crear la base de datos a partir de los datos del enunciado. Como es un ejercico de SQL en Python, se van a importar las funciones de SQL:
import sqlite3

In [ ]:
#Hay que crear tablas e insertar los datos en esas tablas. Vamos paso a paso:

#Conectar a la base de datos (se crea el archivo si no existe):
conexion = sqlite3.connect('distribuidora_cerveza.db')
cursor = conexion.cursor()

#Crear las tablas con sus títulos (1 por grupo de datos):
cursor.execute('''CREATE TABLE IF NOT EXISTS CERVEZAS (CodC TEXT PRIMARY KEY, Envase TEXT, Capacidad REAL, Stock INTEGER)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS BARES (CodB TEXT PRIMARY KEY, Cif TEXT, Nombre TEXT, Localidad TEXT)''')

cursor.execute('''CREATE TABLE IF NOT EXISTS EMPLEADOS (CodE INTEGER PRIMARY KEY, Nombre TEXT, Sueldo INTEGER)''')
#Para la última tabla, aparte de meter los títulos de las columnas, se relaciona con el resto de las tablas mediante los Foreign Keys CodE, CodB y Codc:
cursor.execute('''CREATE TABLE IF NOT EXISTS REPARTO (CodE INTEGER, CodB TEXT, CodC TEXT, Fecha TEXT, Cantidad INTEGER, 
                FOREIGN KEY(CodE) REFERENCES EMPLEADOS(CodE), 
                FOREIGN KEY(CodB) REFERENCES BARES(CodB),
                FOREIGN KEY(CodC) REFERENCES CERVEZAS(CodC))''')

#Insertar los datos de cada grupo:
cervezas = [('01', 'Botella', 0.2, 3600), ('02', 'Botella', 0.33, 1200), ('03', 'Lata', 0.33, 2400), ('04', 'Botella', 1, 288), ('05', 'Barril', 60, 30)]

bares = [('001', '11111111X', 'Stop', 'Villa Botijo'), ('002', '22222222Y', 'Las Vegas', 'Villa Botijo'), ('003', '-', 'Club Social', 'Las Ranas'),
         ('004', '33333333Z', 'Otra Ronda', 'La Esponja')]

empleados = [(1, 'Prudencio Caminero', 120000), (2, 'Vicente Merario', 110000), (3, 'Valentin Siempre', 100000)]

repartos = [(1, '001', '01', '2005-10-21', 240), (1, '001', '02', '2005-10-21', 48), (1, '002', '03', '2005-10-22', 60), (1, '004', '05', '2005-10-22', 4),
            (2, '002', '03', '2005-10-22', 48), (2, '002', '05', '2005-10-23', 2), (2, '004', '01', '2005-10-23', 480), (2, '004', '02', '2005-10-24', 72),
            (3, '003', '03', '2005-10-24', 48), (3, '003', '04', '2005-10-25', 20)]

cursor.executemany('INSERT OR REPLACE INTO CERVEZAS VALUES (?,?,?,?)', cervezas)
cursor.executemany('INSERT OR REPLACE INTO BARES VALUES (?,?,?,?)', bares)
cursor.executemany('INSERT OR REPLACE INTO EMPLEADOS VALUES (?,?,?)', empleados)
cursor.executemany('INSERT OR REPLACE INTO REPARTO VALUES (?,?,?,?,?)', repartos)

#Guardar cambios y cerrar:
conexion.commit()
conexion.close()

print("Base de datos creada y datos insertados con éxito")

Base de datos creada y datos insertados con éxito.


In [ ]:
#Cada enunciado se hará en una función:

def consultar_reparto_stop():
    #Conectar a la base de datos:
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Consulta SQL:
    query = '''SELECT DISTINCT E.Nombre
            FROM EMPLEADOS E
            JOIN REPARTO R ON E.CodE = R.CodE
            JOIN BARES B ON R.CodB = B.CodB
            WHERE B.Nombre = 'Stop' AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23';'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
empleados = consultar_reparto_stop()

print("Empleados que repartieron en 'Stop' (17-23 Oct 2005):")
for emp in empleados:
    print(f"- {emp[0]}")

#Prudencio Caminero (puesto que es el único que realizó entregas al bar "Stop" el día 2005-10-21)

Empleados que repartieron en 'Stop' (17-23 Oct 2005):
- Prudencio Caminero


In [ ]:
#Necesitamos filtrar por el campo Envase y Capacidad de la tabla CERVEZAS, y luego mostrar los datos de la tabla BARES:

def consultar_bares_botella_pequena():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Consulta con JOINs y filtros de envase/capacidad:
    query = '''SELECT DISTINCT B.Cif, B.Nombre, B.Localidad
            FROM BARES B
            JOIN REPARTO R ON B.CodB = R.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            WHERE C.Envase = 'Botella' AND C.Capacidad < 1
            ORDER BY B.Localidad;'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
bares = consultar_bares_botella_pequena()

print(f"{'CIF':<12} | {'Nombre':<15} | {'Localidad'}")
print("-" * 45)
for cif, nombre, localidad in bares:
    print(f"{cif:<12} | {nombre:<15} | {localidad}")

#Otra Ronda (La Esponja) - Recibió botellas de 0.2 y 0.33
#Stop (Villa Botijo) - Recibió botellas de 0.2 y 0.33

CIF          | Nombre          | Localidad
---------------------------------------------
33333333Z    | Otra Ronda      | La Esponja
11111111X    | Stop            | Villa Botijo


In [ ]:
#Necesitamos unir las cuatro tablas (EMPLEADOS, REPARTO, BARES y CERVEZAS) filtrando específicamente por el nombre del empleado:

def consultar_repartos_prudencio():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Consulta uniendo las 4 tablas:
    query = '''SELECT B.Nombre, C.Envase, C.Capacidad, R.Fecha, R.Cantidad
            FROM REPARTO R
            JOIN EMPLEADOS E ON R.CodE = E.CodE
            JOIN BARES B ON R.CodB = B.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            WHERE E.Nombre = 'Prudencio Caminero';'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
repartos = consultar_repartos_prudencio()

print(f"{'Bar':<15} | {'Envase':<10} | {'Cap.':<5} | {'Fecha':<12} | {'Cant.'}")
print("-" * 60)
for bar, envase, cap, fecha, cant in repartos:
    print(f"{bar:<15} | {envase:<10} | {cap:<5} | {fecha:<12} | {cant}")

# Stop: Botella (0.2) el 21/10/05 - 240 unidades
# Stop: Botella (0.33) el 21/10/05 - 48 unidades
# Las Vegas: Lata (0.33) el 22/10/05 - 60 unidades
# Otra Ronda: Barril (60.0) el 22/10/05 - 4 unidades

Bar             | Envase     | Cap.  | Fecha        | Cant.
------------------------------------------------------------
Stop            | Botella    | 0.2   | 2005-10-21   | 240
Stop            | Botella    | 0.33  | 2005-10-21   | 48
Las Vegas       | Lata       | 0.33  | 2005-10-22   | 60
Otra Ronda      | Barril     | 60.0  | 2005-10-22   | 4


In [ ]:
#Filtramos en la tabla CERVEZAS por el tipo de envase y las dos capacidades específicas, vinculándolas con la tabla BARES a través de los repartos:

def consultar_bares_botella_especifica():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Consulta con filtro OR para las capacidades:
    query = '''SELECT DISTINCT B.Nombre, B.Localidad
            FROM BARES B
            JOIN REPARTO R ON B.CodB = R.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            WHERE C.Envase = 'Botella' AND (C.Capacidad = 0.2 OR C.Capacidad = 0.33);'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
bares = consultar_bares_botella_especifica()

print(f"{'Nombre del Bar':<15} | {'Localidad'}")
print("-" * 35)
for nombre, localidad in bares:
    print(f"{nombre:<15} | {localidad}")
    
#Stop (Villa Botijo): Recibió botellas de 0.2 y 0.33
#Otra Ronda (La Esponja): Recibió botellas de 0.2 y 0.33

Nombre del Bar  | Localidad
-----------------------------------
Stop            | Villa Botijo
Otra Ronda      | La Esponja


In [ ]:
#Necesitamos encontrar a los empleados que hayan hecho repartos que cumplan simultáneamente tres condiciones:
#que el bar sea "Stop" o "Las Vegas", que el envase sea "Botella" y que el empleado aparezca en esos registros

def consultar_empleados_reparto_botellas_especificos():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Consulta que une EMPLEADOS, REPARTO, BARES y CERVEZAS:
    query = '''SELECT DISTINCT E.Nombre
            FROM EMPLEADOS E
            JOIN REPARTO R ON E.CodE = R.CodE
            JOIN BARES B ON R.CodB = B.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            WHERE B.Nombre IN ('Stop', 'Las Vegas') AND C.Envase = 'Botella';'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
empleados = consultar_empleados_reparto_botellas_especificos()

print("Empleados que repartieron botellas a 'Stop' o 'Las Vegas':")
for emp in empleados:
    print(f"- {emp[0]}")
    
#Prudencio Caminero: Repartió botellas al bar Stop el 21/10/05.
#Aunque hubo repartos al bar "Las Vegas", fueron de tipo Lata y Barril, por lo que no se incluyen en este filtro de botellas

Empleados que repartieron botellas a 'Stop' o 'Las Vegas':
- Prudencio Caminero


In [ ]:
#Debemos filtrar los repartos donde la Localidad del bar sea distinta a 'Villa Botijo'
#Usaremos la función de agregado COUNT para contar los registros (viajes) y agruparemos por el nombre del empleado:

def consultar_viajes_fuera_villa_botijo():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Agrupamos por empleado y contamos los registros donde la localidad NO es Villa Botijo:
    query = '''SELECT E.Nombre, COUNT(R.CodE) as Numero_Viajes
            FROM EMPLEADOS E
            JOIN REPARTO R ON E.CodE = R.CodE
            JOIN BARES B ON R.CodB = B.CodB
            WHERE B.Localidad <> 'Villa Botijo'
            GROUP BY E.Nombre;'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
viajes = consultar_viajes_fuera_villa_botijo()

print(f"{'Empleado':<20} | {'Viajes fuera de Villa Botijo'}")
print("-" * 50)
for nombre, num_viajes in viajes:
    print(f"{nombre:<20} | {num_viajes}")
    
#Prudencio Caminero: 1 viaje (al bar Otra Ronda en La Esponja)
#Vicente Merario: 2 viajes (al bar Otra Ronda en La Esponja)
#Valentin Siempre: 2 viajes (al bar Club Social en Las Ranas)

Empleado             | Viajes fuera de Villa Botijo
--------------------------------------------------
Prudencio Caminero   | 1
Valentin Siempre     | 2
Vicente Merario      | 2


In [ ]:
#El bar que más litros de cerveza ha comprado es Otra Ronda, ubicado en La Esponja, con un total de 359,76 litros:

def bar_con_mas_litros():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Calculamos la suma de (capacidad * cantidad) para cada bar:
    query = '''SELECT B.Nombre, B.Localidad, SUM(C.Capacidad * R.Cantidad) as TotalLitros
            FROM BARES B
            JOIN REPARTO R ON B.CodB = R.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            GROUP BY B.CodB
            ORDER BY TotalLitros DESC
            LIMIT 1;'''

    cursor.execute(query)
    resultado = cursor.fetchone()
    
    conexion.close()
    return resultado

#Ejecutar y mostrar el bar ganador:
nombre, localidad, litros = bar_con_mas_litros()
print(f"El bar con más compras es '{nombre}' en {localidad} con {litros:.2f} litros.")

El bar con más compras es 'Otra Ronda' en La Esponja con 359.76 litros.


In [13]:
#Primero debemos identificar cuántas cervezas distintas cumplen con ser "Botella" y tener capacidad menor a 1 litro (son 2: la de 0.2 y la de 0.33)
#Luego, filtramos los bares que hayan recibido exactamente esa cantidad de tipos distintos:

def consultar_bares_todas_las_botellas():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Usamos HAVING para comparar la cuenta de tipos distintos recibidos con el total de tipos que existen bajo ese criterio:
    query = '''SELECT B.Nombre
            FROM BARES B
            JOIN REPARTO R ON B.CodB = R.CodB
            JOIN CERVEZAS C ON R.CodC = C.CodC
            WHERE C.Envase = 'Botella' AND C.Capacidad < 1
            GROUP BY B.CodB
            HAVING COUNT(DISTINCT C.CodC) = (SELECT COUNT(*) 
            FROM CERVEZAS WHERE Envase = 'Botella' AND Capacidad < 1);'''

    cursor.execute(query)
    resultados = cursor.fetchall()
    conexion.close()
    return resultados

#Ejecutar y mostrar resultados:
bares = consultar_bares_todas_las_botellas()

print("Bares que han adquirido todos los tipos de botellas pequeñas (<1L):")
for bar in bares:
    print(f"- {bar[0]}")

#Stop y Otra Ronda: recibió los códigos 01 y 02

Bares que han adquirido todos los tipos de botellas pequeñas (<1L):
- Stop
- Otra Ronda


In [14]:
#Para resolver esto, primero identificamos al empleado que tiene más fechas distintas registradas en la tabla REPARTO y luego aplicamos la actualización en la tabla EMPLEADOS
#Como Prudencio Caminero y Vicente Merario empatan con repartos en 2 días distintos, el script actualizará a ambos

def actualizar_sueldo_mayor_actividad():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Identificar el número máximo de días trabajados:
    cursor.execute('''SELECT MAX(conteo) 
                   FROM (SELECT COUNT(DISTINCT Fecha) as conteo 
                    FROM REPARTO 
                    GROUP BY CodE)''')
    max_dias = cursor.fetchone()[0]

    #Actualizar el sueldo (subir 5%) a los empleados que tengan ese máximo de días:
    query_update = '''UPDATE EMPLEADOS
                    SET Sueldo = Sueldo * 1.05
                    WHERE CodE IN (SELECT CodE 
                                    FROM REPARTO 
                                    GROUP BY CodE 
                                    HAVING COUNT(DISTINCT Fecha) = ?);'''
    
    cursor.execute(query_update, (max_dias,))
    filas_afectadas = cursor.rowcount
    
    conexion.commit()
    
    #Consultar nuevos sueldos para verificar:
    cursor.execute("SELECT Nombre, Sueldo FROM EMPLEADOS")
    nuevos_sueldos = cursor.fetchall()
    
    conexion.close()
    return filas_afectadas, nuevos_sueldos

#Ejecutar y mostrar resultados:
filas, sueldos = actualizar_sueldo_mayor_actividad()

print(f"Se ha actualizado el sueldo de {filas} empleado(s).\n")
print(f"{'Nombre':<20} | {'Nuevo Sueldo'}")
print("-" * 35)
for nombre, sueldo in sueldos:
    print(f"{nombre:<20} | {sueldo:,.0f}")

#Prudencio (Días: 21 y 22) y Vicente (Días: 22, 23 y 24; nota: el 22 y 23 cuentan como días trabajados) verán su sueldo incrementado
#Valentín se mantiene igual al haber trabajado solo en fechas consecutivas específicas o menos días distintos

Se ha actualizado el sueldo de 1 empleado(s).

Nombre               | Nuevo Sueldo
-----------------------------------
Prudencio Caminero   | 120,000
Vicente Merario      | 121,275
Valentin Siempre     | 100,000


In [15]:
#Para realizar esta inserción correctamente, primero necesitamos recuperar los códigos (CodE, CodB y CodC) asociados a los nombres y descripciones
#para asegurar que la integridad de la base de datos se mantenga:

def insertar_nuevo_reparto():
    conexion = sqlite3.connect('distribuidora_cerveza.db')
    cursor = conexion.cursor()

    #Obtenemos los códigos necesarios mediante subconsultas:
    #Vicente Merario (CodE), Stop (CodB) y Lata (CodC - asumiendo la lata de 0.33)
    query_insert = '''INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
                    VALUES ((SELECT CodE FROM EMPLEADOS WHERE Nombre = 'Vicente Merario'),
                            (SELECT CodB FROM BARES WHERE Nombre = 'Stop'),
                            (SELECT CodC FROM CERVEZAS WHERE Envase = 'Lata' LIMIT 1),'2005-10-26', 48);'''

    try:
        cursor.execute(query_insert)
        conexion.commit()
        print("Nuevo reparto insertado con éxito.")
    except sqlite3.Error as e:
        print(f"Error al insertar: {e}")
    finally:
        conexion.close()

#Ejecutar la inserción:
insertar_nuevo_reparto()

#Empleado: Vicente Merario (CodE: 2)
#Bar: Stop (CodB: 001)
#Producto: Lata (CodC: 03)
#Fecha: 2005-10-26
#Cantidad: 48 unidades

Nuevo reparto insertado con éxito.
